# Stage 4 — Final Held-Out Test Evaluation (LOCKED C08, seed 456)

**Our final model has already been selected. No training or tuning happens here.**

| Frozen decision | Value |
|---|---|
| Architecture | Stage 3F reference CNN, 128×128 grayscale input |
| Preprocessing | RGB conversion → LANCZOS pad to 128×128 → grayscale → CLAHE `clipLimit=3.0`, `tileGridSize=(8,8)` → divide by 255 |
| Model | **C08, training seed 456** |
| Checkpoint | `stability_v1/C08/seed_456/best_accuracy_model.keras` |
| Checkpoint selection | Highest validation accuracy; among tied epochs, lower validation loss |
| Selected validation performance | **50/52 = 96.15%**, macro F1 **0.9528**, best accuracy epoch **11** |
| Final test | **51 held-out images**, untouched in Stages 1–3F |

**Protocol:** (1) verify the frozen v1 split and exact checkpoint; (2) independently recompute the *validation* metrics for provenance; (3) write a selection lock with hashes; (4) evaluate the 51 test images **once**, then save the final metrics and confusion matrix. Re-running this notebook afterward displays saved results **without predicting on test images again**. An interrupted run deliberately requires manual review rather than silently rerunning the test.

The validation set has been used extensively for tuning and model/seed selection, so **96.15% is not an unbiased final result**. Our small test set gives an image-level estimate only. Patient independence is **unverified**, so this is an educational research prototype, **not a clinical diagnostic system**. All X-rays, paths, predictions, model weights and private metadata must remain private, not on a public GitHub repo.

**Run:** In a fresh Colab runtime, choose a GPU if available (CPU also works). Use **Runtime → Run all**, and authorize access to the same Google Drive as your earlier notebooks. The notebook does not install or require third-party packages beyond the standard Colab scientific environment.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports and immutable experiment paths

The selected checkpoint is the **seed-456 C08 model from Stage 3F**, not the seed-42 C08-R1 model.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, log_loss,
)
import tensorflow as tf

DATA_ROOT = Path('/content/drive/MyDrive/224')
PROJECT_ROOT = Path('/content/drive/MyDrive/Spinal_Classification_Optimization')
SPLIT_DIR = PROJECT_ROOT / 'splits/v1'
STAGE3_DIR = PROJECT_ROOT / 'experiments/stage3'
SELECTED_DIR = STAGE3_DIR / 'stability_v1/C08/seed_456'
CHECKPOINT_PATH = SELECTED_DIR / 'best_accuracy_model.keras'
CONFIG_PATH = SELECTED_DIR / 'configuration.json'
VAL_METRICS_PATH = SELECTED_DIR / 'validation_metrics.json'

FINAL_DIR = PROJECT_ROOT / 'final_evaluation/v1'
LOCK_PATH = FINAL_DIR / 'selection_LOCKED_PRIVATE.json'
ATTEMPT_PATH = FINAL_DIR / 'test_attempt_PRIVATE.json'
RESULT_PATH = FINAL_DIR / 'final_test_metrics_PRIVATE.json'

CLASS_TO_LABEL = {'Normal': 0, 'Scol': 1, 'Spond': 2}
LABEL_TO_NAME = {0: 'Normal', 1: 'Scoliosis', 2: 'Spondylolisthesis'}
SPLIT_SEED = 42
EXPECTED_SEED = 456
BATCH_SIZE = 16
PREPROCESSING = {
    'mode': 'clahe', 'clip_limit': 3.0,
    'tile_grid': [8, 8], 'size': 128,
    'pixel_normalization': 'divide by 255.0 after CLAHE',
}

print('TensorFlow:', tf.__version__, '| OpenCV:', cv2.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Selected checkpoint:', CHECKPOINT_PATH)
print('Private final evaluation directory:', FINAL_DIR)
assert CHECKPOINT_PATH.is_file(), 'Selected model not found. Check Google Drive mount.'
assert CONFIG_PATH.is_file() and VAL_METRICS_PATH.is_file(), 'Stage 3F saved metadata missing.'

## 3. Verify the original frozen dataset split

Metadata for all 338 images is checked, but **no test images are decoded or predicted on here**. We verify file identities later when evaluating the reserved set.

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest_PRIVATE.csv'
split_config_path = SPLIT_DIR / 'split_config_PRIVATE.json'
assert manifest_path.is_file() and split_config_path.is_file(), 'Frozen Stage 2 split missing.'
manifest = pd.read_csv(manifest_path, keep_default_na=False)
freeze = json.loads(split_config_path.read_text(encoding='utf-8'))

required = {'relative_path', 'class_name', 'label', 'file_sha256', 'group_id', 'split'}
assert required <= set(manifest.columns)
assert len(manifest) == 338 and manifest['relative_path'].is_unique
assert freeze['seed'] == SPLIT_SEED
assert freeze['class_to_label'] == CLASS_TO_LABEL
assert freeze['patient_independent'] is False
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert manifest.groupby('group_id')['split'].nunique().max() == 1
assert all(int(r.label) == CLASS_TO_LABEL[r.class_name]
           for r in manifest.itertuples(index=False))

fingerprint_rows = (manifest[['relative_path', 'class_name', 'label', 'file_sha256']]
                    .sort_values('relative_path')
                    .to_dict(orient='records'))
fingerprint = hashlib.sha256(
    json.dumps(fingerprint_rows, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
assert fingerprint == freeze['dataset_fingerprint'], 'Dataset fingerprint changed! STOP.'

counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=list(CLASS_TO_LABEL), columns=['train', 'validation', 'test'], fill_value=0
)
display(counts.assign(total=counts.sum(axis=1)))
assert counts['train'].sum() == 235
assert counts['validation'].sum() == 52
assert counts['test'].sum() == 51

train_df = manifest[manifest['split'] == 'train'].sort_values('relative_path').reset_index(drop=True)
val_df = manifest[manifest['split'] == 'validation'].sort_values('relative_path').reset_index(drop=True)
test_df = manifest[manifest['split'] == 'test'].sort_values('relative_path').reset_index(drop=True)
assert set(train_df['relative_path']).isdisjoint(test_df['relative_path'])
assert set(val_df['relative_path']).isdisjoint(test_df['relative_path'])
print('Frozen split verified. The 51 test X-rays have NOT been read.')

## 4. Verify the exact Stage 3F checkpoint and saved validation report

This step stops on any mismatch in model identity, saved selection policy, dataset fingerprint, or class count. The checkpoint hash becomes part of our immutable final evaluation record.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

selected_config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
selected_metrics = json.loads(VAL_METRICS_PATH.read_text(encoding='utf-8'))
expected_config = {
    'stage': '3F', 'candidate': 'C08', 'model_seed': EXPECTED_SEED,
    'split_seed': SPLIT_SEED, 'dataset_fingerprint': fingerprint,
    'split_version': 'v1', 'architecture': 'reference',
    'input_mode': 'clahe', 'clip_limit': 3.0, 'tile_grid': [8, 8],
    'epochs_max': 40, 'batch_size': BATCH_SIZE,
    'learning_rate': 3e-4,
    'checkpoint_policy': 'best_val_loss_and_best_val_accuracy_loss_tiebreak',
    'parameter_count': 4287491,
    'patient_independent': False, 'test_evaluated': False,
}
for key, expected in expected_config.items():
    assert selected_config.get(key) == expected, (
        f'Wrong selected model setting {key}: {selected_config.get(key)!r}'
    )
assert selected_metrics.get('candidate') == 'C08'
assert selected_metrics.get('model_seed') == EXPECTED_SEED
assert selected_metrics.get('test_evaluated') is False
saved_val = selected_metrics['best_accuracy']
assert saved_val['correct'] == 50
assert abs(saved_val['val_accuracy'] - 50 / 52) < 1e-5
assert abs(saved_val['val_macro_f1'] - 0.95283) < 0.001
assert saved_val['selected_epoch'] == 11
assert 0.39 < saved_val['val_loss'] < 0.42
assert not RESULT_PATH.exists() or LOCK_PATH.is_file(), (
    'A result exists without a lock. Stop and inspect manually.'
)

checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
print('Checkpoint SHA256:', checkpoint_sha256)
print('Historical validation: 50/52 = 96.15%, macro F1',
      round(saved_val['val_macro_f1'], 4), 'at epoch', saved_val['selected_epoch'])
print('Source TensorFlow:', selected_config.get('tensorflow_version'))
print('Current TensorFlow:', tf.__version__)
if selected_config.get('tensorflow_version') != str(tf.__version__):
    print('WARNING: Different TensorFlow version. Validation verification below must still match.')

## 5. Reuse the exact preprocessing pipeline

These functions match Stage 3F. No transformations are fitted using test data; CLAHE uses fixed settings independently on each image. We verify the file's SHA256 against the frozen manifest before decoding it.

In [ ]:
def preprocess_clahe(path):
    with Image.open(path) as opened:
        rgb_image = ImageOps.exif_transpose(opened).convert('RGB')
        resized = ImageOps.pad(
            rgb_image, (128, 128), method=Image.Resampling.LANCZOS,
            color=(0, 0, 0)
        )
        rgb = np.asarray(resized, dtype=np.uint8)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    enhanced = cv2.createCLAHE(
        clipLimit=3.0, tileGridSize=(8, 8)
    ).apply(gray)
    return enhanced[..., None].astype(np.float32) / 255.0


def load_frozen_partition(rows):
    X, y = [], []
    for item in rows.itertuples(index=False):
        image_path = DATA_ROOT / item.relative_path
        if not image_path.is_file():
            raise RuntimeError(f'Missing frozen image: {item.relative_path}')
        if sha256_file(image_path) != item.file_sha256:
            raise RuntimeError(f'Changed frozen image: {item.relative_path}')
        X.append(preprocess_clahe(image_path))
        y.append(int(item.label))
    X = np.stack(X)
    y = np.asarray(y, dtype=np.int64)
    assert X.shape == (len(rows), 128, 128, 1)
    assert np.isfinite(X).all() and (0 <= X.min() <= X.max() <= 1)
    return X, y


def predict_labels(model, X):
    probabilities = np.asarray(model.predict(X, batch_size=BATCH_SIZE, verbose=0))
    assert probabilities.shape == (len(X), 3)
    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)
    return probabilities, np.argmax(probabilities, axis=1)

print('Preprocessing ready. Test images not loaded.')

## 6. Independently confirm validation performance **before** locking

This is an identity/integrity check of the already-selected model, **not a new selection round**. Only 52 validation images are loaded here. If metrics do not match the saved experiment, stop and investigate before the final test.

In [ ]:
model = tf.keras.models.load_model(str(CHECKPOINT_PATH), compile=False)
assert model.count_params() == 4287491
assert tuple(model.input_shape[1:]) == (128, 128, 1)
assert tuple(model.output_shape[1:]) == (3,)

X_val, y_val = load_frozen_partition(val_df)
val_probabilities, val_pred = predict_labels(model, X_val)
val_acc = float(accuracy_score(y_val, val_pred))
val_f1 = float(f1_score(y_val, val_pred, labels=[0, 1, 2],
                        average='macro', zero_division=0))
val_cm = confusion_matrix(y_val, val_pred, labels=[0, 1, 2])
assert int((val_pred == y_val).sum()) == 50, 'Selected checkpoint failed validation reproduction.'
assert abs(val_acc - saved_val['val_accuracy']) < 1e-6
assert abs(val_f1 - saved_val['val_macro_f1']) < 1e-6
assert val_cm.tolist() == saved_val['confusion_matrix']
print('VALIDATION INTEGRITY CHECK PASSED')
print(f'Accuracy: {val_acc:.2%} (50/52); macro F1: {val_f1:.4f}')
display(pd.DataFrame(val_cm, index=list(LABEL_TO_NAME.values()),
                     columns=list(LABEL_TO_NAME.values())))
del X_val, val_probabilities
print('No test images loaded or predictions produced yet.')

## 7. Write and verify the **selection lock** (before any test image decoding)

A SHA256 digest binds the selected model file to the exact frozen dataset manifest. If the lock already exists, its contents must match. This step never overwrites it.

In [ ]:
selection_identity = {
    'project': 'Spinal Disorder Classification',
    'stage': '4',
    'split_version': 'v1',
    'dataset_fingerprint': fingerprint,
    'patient_independent': False,
    'selected_candidate': 'C08',
    'model_seed': 456,
    'checkpoint_selection_rule': 'highest validation accuracy, lower validation loss tie-break',
    'checkpoint_path': str(CHECKPOINT_PATH),
    'checkpoint_sha256': checkpoint_sha256,
    'model_parameters': 4287491,
    'preprocessing': PREPROCESSING,
    'validation_correct': 50,
    'validation_n': 52,
    'validation_accuracy': float(saved_val['val_accuracy']),
    'validation_macro_f1': float(saved_val['val_macro_f1']),
    'validation_loss_at_selected_epoch': float(saved_val['val_loss']),
    'selected_epoch': int(saved_val['selected_epoch']),
    'final_test_n': 51,
    'final_test_policy': 'single locked test evaluation, no post-test model changes',
}
FINAL_DIR.mkdir(parents=True, exist_ok=True)
if LOCK_PATH.exists():
    old_lock = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
    for k, v in selection_identity.items():
        if old_lock.get(k) != v:
            raise RuntimeError(f'Existing Stage 4 model-selection LOCK differs on {k}; STOP.')
    print('Existing selection LOCK verified; not overwritten.')
else:
    if ATTEMPT_PATH.exists() or RESULT_PATH.exists():
        raise RuntimeError('Prior test attempt exists without a selection lock; STOP.')
    new_lock = {
        **selection_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    with LOCK_PATH.open('x', encoding='utf-8') as stream:
        json.dump(new_lock, stream, indent=2)
    print('NEW SELECTION LOCK WRITTEN before test decoding.')
print('Locked candidate: C08 | model seed: 456 | checkpoint: best_accuracy_model.keras')
print('Selection file:', LOCK_PATH)

## 8. The one-time held-out evaluation

**Run this section only after the preceding validation and lock checks have passed.**

The code writes an `attempt` record **before** opening a test image. If a complete result already exists, it loads that **saved JSON** and does not load or predict the test set. If an attempt exists without a completed result, it stops for manual investigation rather than evaluating the test set again.

The final test includes **all 51 images** exactly once. Do not alter the model, preprocessing, or checkpoint based on these results.

In [ ]:
def wilson_interval(correct, n, z=1.959963984540054):
    # Descriptive 95% binomial confidence interval; does not correct patient overlap.
    p = correct / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return [float(max(0, center-radius)), float(min(1, center+radius))]


def evaluate_test_once():
    if not LOCK_PATH.is_file():
        raise RuntimeError('Model selection must be locked before testing.')
    saved_lock = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
    assert saved_lock['checkpoint_sha256'] == checkpoint_sha256
    assert saved_lock['dataset_fingerprint'] == fingerprint

    if RESULT_PATH.exists():
        result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
        if result.get('checkpoint_sha256') != checkpoint_sha256:
            raise RuntimeError('Existing final result belongs to a different model; STOP.')
        if result.get('dataset_fingerprint') != fingerprint:
            raise RuntimeError('Existing final result belongs to a different split; STOP.')
        print('FINAL TEST ALREADY COMPLETED. Loading stored metrics only.')
        print('No test images loaded; no new predictions made.')
        return result

    if ATTEMPT_PATH.exists():
        raise RuntimeError(
            'A prior final-test attempt exists but no complete result is present. '
            'STOP and inspect outputs manually; do not rerun the held-out test blindly.'
        )

    attempt = {
        'started_at_utc': datetime.now(timezone.utc).isoformat(),
        'status': 'started', 'test_n': 51,
        'checkpoint_sha256': checkpoint_sha256,
        'dataset_fingerprint': fingerprint,
    }
    with ATTEMPT_PATH.open('x', encoding='utf-8') as stream:
        json.dump(attempt, stream, indent=2)

    # This is the FIRST image decoding of the frozen 51-image test partition.
    X_test, y_test = load_frozen_partition(test_df)
    assert len(X_test) == 51 and len(y_test) == 51
    probabilities, predictions = predict_labels(model, X_test)
    assert np.bincount(y_test, minlength=3).tolist() == [11, 28, 12]

    correct = int((predictions == y_test).sum())
    matrix = confusion_matrix(y_test, predictions, labels=[0, 1, 2])
    report = classification_report(
        y_test, predictions, labels=[0, 1, 2],
        target_names=[LABEL_TO_NAME[i] for i in range(3)],
        output_dict=True, zero_division=0,
    )
    metrics = {
        'stage': '4', 'evaluation_type': 'once-only, frozen 51-image test',
        'completed_at_utc': datetime.now(timezone.utc).isoformat(),
        'selected_candidate': 'C08', 'model_seed': 456,
        'checkpoint_filename': 'best_accuracy_model.keras',
        'checkpoint_sha256': checkpoint_sha256,
        'dataset_fingerprint': fingerprint, 'split_version': 'v1',
        'preprocessing': PREPROCESSING,
        'test_images': 51, 'correct_predictions': correct,
        'test_accuracy': float(accuracy_score(y_test, predictions)),
        'test_accuracy_wilson_95ci': wilson_interval(correct, 51),
        'test_macro_f1': float(f1_score(
            y_test, predictions, labels=[0, 1, 2], average='macro', zero_division=0
        )),
        'test_balanced_accuracy': float(balanced_accuracy_score(y_test, predictions)),
        'test_cross_entropy': float(log_loss(y_test, probabilities, labels=[0, 1, 2])),
        'confusion_matrix': matrix.tolist(),
        'classification_report': report,
        'per_class_recall': {
            name: float(report[name]['recall']) for name in LABEL_TO_NAME.values()
        },
        'patient_independent': False,
        'note': 'Image-level held-out evaluation, not clinically validated; patient identity unknown.',
    }

    # Row-level predictions remain PRIVATE; pseudonymous image IDs, no filenames.
    private_rows = []
    for item, truth, pred, probs in zip(
        test_df.itertuples(index=False), y_test, predictions, probabilities
    ):
        image_id = hashlib.sha256(item.relative_path.encode('utf-8')).hexdigest()[:20]
        private_rows.append({
            'image_id_PRIVATE': image_id,
            'true_class_PRIVATE': LABEL_TO_NAME[int(truth)],
            'predicted_class_PRIVATE': LABEL_TO_NAME[int(pred)],
            'correct': int(truth == pred),
            'prob_normal': float(probs[0]),
            'prob_scoliosis': float(probs[1]),
            'prob_spondylolisthesis': float(probs[2]),
        })
    pd.DataFrame(private_rows).to_csv(FINAL_DIR / 'per_image_predictions_PRIVATE.csv', index=False)
    np.save(FINAL_DIR / 'confusion_matrix_test_PRIVATE.npy', matrix)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(matrix, cmap='Blues', vmin=0)
    ax.set_xticks(range(3), [LABEL_TO_NAME[i] for i in range(3)], rotation=24, ha='right')
    ax.set_yticks(range(3), [LABEL_TO_NAME[i] for i in range(3)])
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('Actual class')
    ax.set_title('Locked C08 seed 456: final held-out test')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center',
                    color='white' if matrix[i,j] > matrix.max()/2 else 'black')
    fig.colorbar(im, ax=ax, fraction=.045, pad=.04)
    fig.tight_layout()
    fig.savefig(FINAL_DIR / 'confusion_matrix_test_PRIVATE.png', dpi=175)
    plt.show()
    plt.close(fig)

    # IMPORTANT: final results written LAST so complete JSON is the success marker.
    with RESULT_PATH.open('x', encoding='utf-8') as stream:
        json.dump(metrics, stream, indent=2)
    print('ONE-TIME FINAL TEST COMPLETE. Saved all results privately.')
    return metrics

final_result = evaluate_test_once()

## 9. Read the final result and create a concise, private research summary

This section only reads the **saved result JSON**. It never evaluates or predicts on test images. Share the aggregate metrics and confusion matrix with me; keep row-level predictions private.

In [ ]:
assert RESULT_PATH.is_file(), 'Final test result not available.'
summary = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
assert summary['checkpoint_sha256'] == checkpoint_sha256
assert summary['dataset_fingerprint'] == fingerprint

print('FINAL HELD-OUT RESULTS (51 images)')
print(f"Accuracy: {summary['correct_predictions']}/51 = {summary['test_accuracy']:.2%}")
print(f"Macro F1: {summary['test_macro_f1']:.4f}")
print(f"Balanced accuracy: {summary['test_balanced_accuracy']:.4f}")
print(f"Cross-entropy: {summary['test_cross_entropy']:.4f}")
print('95% Wilson accuracy interval:',
      tuple(f'{x:.2%}' for x in summary['test_accuracy_wilson_95ci']))
print('Per-class recall:', summary['per_class_recall'])
cm = np.asarray(summary['confusion_matrix'], dtype=int)
display(pd.DataFrame(cm, index=[LABEL_TO_NAME[i] for i in range(3)],
                     columns=[LABEL_TO_NAME[i] for i in range(3)]))

val_vs_test = pd.DataFrame([
    {'partition': 'Validation (model selected here)',
     'n_images': 52, 'accuracy': saved_val['val_accuracy'],
     'macro_f1': saved_val['val_macro_f1']},
    {'partition': 'Held-out test (evaluated once)',
     'n_images': 51, 'accuracy': summary['test_accuracy'],
     'macro_f1': summary['test_macro_f1']},
])
display(val_vs_test.round(4))
val_vs_test.to_csv(FINAL_DIR / 'validation_vs_test_summary_PRIVATE.csv', index=False)

per_class_table = pd.DataFrame(summary['classification_report']).T
per_class_table.to_csv(FINAL_DIR / 'test_classification_report_PRIVATE.csv')
display(per_class_table.round(4))

report_text = f"""# Final held-out research evaluation — Stage 4

**Model:** C08; reference CNN; seed 456; best-validation-accuracy checkpoint.  
**Preprocessing:** 128×128 grayscale, CLAHE clip limit 3.0, 8×8 grid, pixels divided by 255.  
**Frozen dataset:** 338 X-rays; 235 training, 52 validation, 51 test.  
**Selection:** chosen before reading any test X-ray.

## Locked validation set (used for selection)
- Accuracy: 50/52 = {saved_val['val_accuracy']:.2%}
- Macro F1: {saved_val['val_macro_f1']:.4f}

## Held-out test set (single evaluation)
- Accuracy: {summary['correct_predictions']}/51 = {summary['test_accuracy']:.2%}
- Macro F1: {summary['test_macro_f1']:.4f}
- Balanced accuracy: {summary['test_balanced_accuracy']:.4f}
- Cross-entropy: {summary['test_cross_entropy']:.4f}
- Descriptive 95% Wilson interval for accuracy: {summary['test_accuracy_wilson_95ci'][0]:.2%} to {summary['test_accuracy_wilson_95ci'][1]:.2%}

## Per-class recall
- Normal: {summary['per_class_recall']['Normal']:.2%}
- Scoliosis: {summary['per_class_recall']['Scoliosis']:.2%}
- Spondylolisthesis: {summary['per_class_recall']['Spondylolisthesis']:.2%}

## Limitations
All results are **image-level**, from a small historical dataset. Patient identity and patient-independent splitting could not be verified. Multiple experiments and seeds were compared on the same validation set, which creates selection bias. Test uncertainty is substantial. Accuracy on this dataset is **not evidence of general clinical safety or diagnostic validity**. This is an educational research demonstration, not a medical device.

## Reproducibility
- Dataset fingerprint: `{fingerprint}`
- Checkpoint SHA256: `{checkpoint_sha256}`
- Original model checkpoint: `{CHECKPOINT_PATH}` (private)
- Final result JSON: `{RESULT_PATH}` (private)
"""
summary_path = FINAL_DIR / 'final_research_summary_PRIVATE.md'
summary_path.write_text(report_text, encoding='utf-8')
print('Private report:', summary_path)
print('Do NOT put model weights, patient images, row-level predictions or private manifests on public GitHub.')

## After Stage 4

1. **Send the final aggregate accuracy, macro F1, and confusion matrix** back in chat. We can discuss what they imply without using the test set for any additional model search.
2. **No post-test tuning:** Do not rerun models, change preprocessing, change the checkpoint, or rank additional candidates based on these 51 images. If future research requires more optimization, obtain a **new external test set** or prospectively collected data with patient-level grouping.
3. **Stage 5:** Organize the GitHub repository, build reproducible documentation, and include honest validation/test results and limitations. Keep private X-rays, personal identifiers, row-level predictions and trained weights out of public repositories unless licensing/privacy has been formally resolved.
4. **Stages 6–7:** Build and deploy a clearly labeled experimental web demo, not a diagnostic tool.

**Do not treat any reported accuracy as patient-independent or clinically validated.**